# D10：独立清洗练习

数据：D:/Learning/study/d10/data/prices_original.csv 和 prices_dirty.csv。
输出：D:/Learning/study/d10/outputs/。
内核选择 Python (D Learning)。先独立写，再与 reference.ipynb 对照。
可以从已准备的 prices_dirty.csv 开始；副本造异常的方法也在参考版中。


1. 将截图里的 lambda 改写成一个具名函数，解释每组 x 的类型及 sum 的统计含义。


In [ ]:
# 在这里独立写代码；运行前先预测记录数量


2. 读取原始八行数据，在副本中加入五条指定记录；或读取准备好的十三行脏样本。保留原始文字。


In [20]:
# 在这里独立写代码；运行前先预测记录数量
from pathlib import Path 
import pandas as pd 
data_dir = Path("D:/Learning/study/d10/data")

original = pd.read_csv(
    data_dir / "prices_original.csv",
    dtype = "string",
    keep_default_na = False,        # 空字段保持为空字符串
)
original_columns = original.columns.tolist()
raw = original.copy()
# len(raw) 是当前行数；本例用它在连续默认索引后追加一行
raw.loc[len(raw)] = ["109", "2026-10-16", "A", "", "1400"]
raw.loc[len(raw)] = ["110", "2026-10-16", "B", "not-a-number", "1000"]
raw.loc[len(raw)] = ["111", "2026-02-30", "A", "109", "1500"]
# 完整复制原来的第一条记录，ignore_index=True 重新生成默认索引
raw = pd.concat([raw, original.iloc[[0]].copy()], ignore_index=True)


raw.loc[len(raw)] = ["112", "2026-10-13", "A", "999", "1200"]
raw = raw.astype("string")

print(original.shape)  # (8, 5)
print(raw.shape)    #(13,5)



(8, 5)
(13, 5)


3. 新增 source_row、price_clean、date_clean；解释为什么不覆盖原始日期和价格。


In [22]:
# 在这里独立写代码；运行前先预测记录数量
work = raw.copy()
# source_row：输入数据行序号，从 1 开始，不包含 CSV 表头
work["source_row"] = range(1,len(work) + 1)

work["price_clean"] = pd.to_numeric(
    work["price"].str.strip(),
    errors = "coerce",
)
work["date_clean"] = pd.to_datetime(
    work["date"].str.strip(),
    format = "%Y-%m-%d",
    errors = "coerce"
)

print(work[["price_clean","date_clean"]].isna().sum())

price_clean    2
date_clean     1
dtype: int64


4. 分别识别空价格、非数字价格、非法日期，给出问题说明；预计无效记录 3 行。


In [12]:
# 在这里独立写代码；运行前先预测记录数量
# 原始内容为空

empty_price = work["price"].str.strip().eq("")    ## .eq("")：判断是否等于空字符串
# 原始内容非空，但转不成数字
bad_number = work["price_clean"].isna() & ~empty_price
# 日期转换失败
bad_date = work["date_clean"].isna()


work["issue"] = "" 
for mask , reason in [
    (empty_price, "空价格；"),
    (bad_number, "非数字价格；"),
    (bad_date, "非法日期；"),
]:
    work.loc[mask,"issue"] = work.loc[mask,"issue"] + reason

invalid_mask = empty_price | bad_number | bad_date
invalid = work.loc[invalid_mask].copy()  # 有问题的记录
valid = work.loc[~invalid_mask].copy()   # 继续检查的记录

print(invalid[["source_row", "record_id", "date", "price", "issue"]])
print(len(invalid))  # 3
print(len(valid))    # 10

    source_row record_id        date         price   issue
8            9       109  2026-10-16                  空价格；
9           10       110  2026-10-16  not-a-number  非数字价格；
10          11       111  2026-02-30           109   非法日期；
3
10


5. 用原始五列判断完全重复。比较 keep=False 与 keep='first'；预计仅移除多余副本 1 行。


In [17]:
# 先标记 重复数据  定义+设置比较范围+方法======后续在操作
extra_copy_mask = valid.duplicated(
    subset=original_columns,  # 只比较原始五列
    keep="first",             # 第一条不标记，后续相同副本标记为 True
)
duplicates = valid.loc[extra_copy_mask].copy()
duplicates["issue"] = "完全重复副本：保留同内容第一条"

unique=valid.loc[~extra_copy_mask].copy()

print(duplicates[["source_row", "record_id"]])  # 数据行 12，编号 101
print(len(duplicates))  # 1
print(len(unique))      # 9




    source_row record_id
11          12       101
1
9


6. 按资产及转换后的日期分组，检查不同价格的数量；隔离冲突组的全部记录，预计 2 行。


In [29]:
# 在这里独立写代码；运行前先预测记录数量
grouped_prices = unique.groupby(
    ["asset", "date_clean"]
)["price_clean"]  # 按“资产＋日期”分组，关注价格列
price_count = grouped_prices.transform("nunique")  #将每组的数量对应回组内每一行。

conflict_mask = price_count > 1

conflicts = unique.loc[conflict_mask].copy()
conflicts["issue"] = "同资产同日期价格冲突：待核实"

clean_work = unique.loc[~conflict_mask].copy()

print(conflicts[["record_id", "asset", "date_clean", "price_clean"]])
print(len(conflicts))   # 2
print(len(clean_work))  # 7



   record_id asset date_clean  price_clean
2        103     A 2026-10-13        105.0
12       112     A 2026-10-13        999.0
2
7


7. 合并六行异常，保留原始值、转换值、数据行序号和说明；另存七行清洗结果。


In [30]:
output_dir = Path("D:/Learning/study/d10/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# 合并三类移出记录，保留原始值、转换值、来源行和问题说明
anomalies = pd.concat(
    [invalid, duplicates, conflicts],
    ignore_index=True,
).sort_values("source_row")



# 清洗结果采用转换后的日期和价格，同时保留来源行
clean = clean_work[[
    "source_row", "record_id", "date_clean",
    "asset", "price_clean", "volume"
]].rename(columns={
    "date_clean": "date",
    "price_clean": "price",
})

clean = clean.sort_values(["asset", "date"]).reset_index(drop=True)

anomalies.to_csv(
    output_dir / "anomalies.csv",
    index=False,
    encoding="utf-8-sig",
)

clean.to_csv(
    output_dir / "prices_clean.csv",
    index=False,
    encoding="utf-8-sig",
)

print(len(anomalies))  # 6
print(len(clean))      # 7


6
7


8. 核对输入和输出行数：13 = 3 + 1 + 2 + 7。解释为什么清洗结果不是原来的八行。


In [ ]:
# 在这里独立写代码；运行前先预测记录数量


9. 从空白复做 217，补充无重复、所有值相同等情况；解释平均时间复杂度与额外空间。


In [ ]:
# 在这里独立写代码；运行前先预测记录数量
